In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  


In [3]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started!

In [4]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [5]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Nice to meet you. I’m ChatGPT, here to help with questions, ideas, writing, planning, coding, or just a friendly chat.\n\nWhat would you like to do today? If you’re not sure, here are a few ideas:\n- Learn something new with a quick explanation\n- Get help writing (email, essay, story, resume)\n- Brainstorm ideas or plan something (a trip, a project, a party)\n- Solve a problem or debug code\n- Practice a language or have a casual chat\n- Play a quick word game or trivia\n\nTell me a bit about your interests or what you’re hoping to do, and we’ll go from there!'

## OK onwards with our first project

In [6]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [7]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [8]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

## And now let's build useful messages for GPT-4.1-mini, using a function

In [10]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [11]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Time to bring it together - the API for OpenAI is very simple!

In [12]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [13]:
summarize("https://edwarddonner.com")

'# Edward Donner’s Playground of AI, Code & Ego\n\nWelcome to Ed’s digital lair, where he casually flexes his credentials as a serial AI startup founder, ex-JPMorgan Managing Director, and LLM evangelist. This guy loves coding, “very amateur” electronic music, and pretending to understand Hacker News.\n\nNot content with just geeking out, Ed’s got a suite of best-selling Udemy courses on AI that have apparently hooked 600,000 students worldwide—surprising even him. For the socially awkward, you can even chat with his digital avatar (because why not?).\n\n**Newsworthy tidbits:**  \n- Dropped shiny new AI courses recently on coding agents, voice AI, AI MLOps, and how to pick the order to devour them.  \n- Runs something called “Outsmart” where language models duke it out in a diplomatic free-for-all (or devious brawl, same thing).  \n\nEmail him if you want to nerd out or just need more AI in your life. LinkedIn, Twitter, Facebook links included—because Ed is everywhere.'

In [14]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [15]:
display_summary("https://edwarddonner.com")

# Edward Donner’s Personal Playground for AI Geeks and Code Nerds

So, you found Ed’s corner of the internet. He’s a coder who loves LLMs so much he lectures friends until they run away, then unleashes those ramblings into Udemy courses that 600,000 people *shockingly* love. He’s also a serial AI startup founder and ex-JPMorgan bigwig, now CTO at Nebula.io.

If you dig AI battles where models fight in the mud of diplomacy and trickery (think LLM Game of Thrones), check out his “Outsmart” arena. He drops regular AI course updates and resource dumps so you can climb the shiny mystical ladder from “Vibe Coder” to “Agentic Engineer.” Also, he’s dabbled in amateur electronic music and pretends Hacker News is his second home.

Want to chat with his digital avatar or just marvel at his AI obsession? That’s all here, neatly served with a side of nerdy charm.

**Latest News:**  
- Feb 17, 2026: New resources on leveling up from AI Coder to Agentic Engineer  
- Jan 4, 2026: AI Builder with n8n — create (voice) agents (or so he says)  
- Sept 15, 2025: AI Engineering MLOps course resources dropped  
- May 28, 2025: Guidance on the perfect AI course binge order  

Basically: if AI and code are your jam, Ed’s site is your candy store.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [16]:
display_summary("https://cnn.com")

# CNN: Because You Clearly Wanted More News Than You Can Handle

Welcome to the sprawling news jungle known as CNN! They’ve got *everything*: US drama, world chaos, political circus, health tips, celebrity gossip, tech gadgets, and even down-to-earth stuff like pets and outdoors because apparently, you need a full life while digesting all this info.

**Breaking News & Hot Topics:**  
- World Cup 2026 hype  
- Russia-Ukraine drama  
- Israel-Hamas conflict  
- Amazon Prime Day—because news without shopping discounts isn’t real news  

**User Experience Highlights:**  
- Constantly asking if their ads annoyed you (Spoiler: they probably did)  
- Video player with all the charm of a lazy sloth  
- Newsletter sign-ups and account struggles for maximum user engagement headaches  

In short, if you want your daily dose of existential dread, excitement about the next soccer World Cup, and a never-ending parade of ads and feedback pop-ups, CNN is your go-to. Just don’t blame us if you end up overwhelmed or just hitting “Cancel” on their relentless ad feedback surveys.

In [17]:
display_summary("https://anthropic.com")

# Anthropic: AI Safety Geeks Assemble! 🤖🦸‍♂️

Anthropic is basically your friendly neighborhood AI safety squad—public benefit corporation style—making sure AI doesn’t turn into a total disaster. They do *a lot* of research, policy work, and offer a lineup of AI products with names straight out of a poetry slam: Mythos, Fable, Opus, Sonnet, Haiku. Because why not?

They have *Claude*, their AI superstar, available in flavors like Claude Code and Claude Cowork (yes, even AI wants to be a team player).

### Latest Drama (aka News & Announcements):
- **Project Glasswing Expansion:** Now reaching 150 organizations across 15+ countries as of June 2, 2026. Because AI transparency isn’t just a buzzword.
- **Claude Opus 4.8:** A fresh upgrade to their Opus model, probably making it less likely to fluff things up.

In short: If you’re into AI that’s *actually* trying not to wreck the future, Anthropic is your crew. They juggle research, policy stances, developer resources, and shiny new AI toys to prove it.